Parsing results

In [7]:
import os
import re
import json
from pathlib import Path
from bs4 import BeautifulSoup
import csv

In [8]:
def parse_readvars_csv(csv_path: str | Path) -> dict:
    """Parse EnergyPlus ReadVars CSV (eplusout.csv) into a dict of timeseries.

    Heuristics used:
    - Skip the first header row(s) until the line that starts with "" (blank first cell)
      followed by variable columns (older ReadVars sometimes include metadata rows).
    - Detect whether the CSV is hourly by counting rows (>= 8000 rows is treated as yearly hourly).
    - Return a dict mapping sanitized variable names to numeric arrays.

    This implementation is conservative: it reads numeric columns only and limits
    memory by only reading the first ~20000 rows.
    """
    csv_path = Path(csv_path)
    series = {}
    try:
        with csv_path.open('r', encoding='utf-8', errors='replace') as fh:
            reader = csv.reader(fh)
            rows = []
            # Read all rows up to a reasonable cap (protect memory)
            max_rows = 20000
            for i, row in enumerate(reader):
                rows.append(row)
                if i + 1 >= max_rows:
                    break

        if len(rows) < 2:
            return {}

        # Attempt to find the header row which contains variable names
        header_idx = 0
        # Common EnergyPlus ReadVars CSV has first column blank then variable labels
        for idx, r in enumerate(rows[:10]):
            # Heuristic: header row contains at least 2 non-empty cells and not numeric
            non_empty = sum(1 for c in r if c and c.strip() != '')
            if non_empty >= 2 and any(not _is_number_like(c) for c in r):
                header_idx = idx
                break

        header = rows[header_idx]
        data_rows = rows[header_idx+1:]

        # If the first column is empty and second column looks like a number for the
        # first data row, then columns from 1..N are variables
        col_count = len(header)
        if col_count < 2:
            return {}

        # Determine if this looks like an hourly file (approx 8760 rows)
        hourly_like = len(data_rows) >= 8000

        # For each column, try to parse numeric values
        for col_idx in range(1, col_count):
            var_name_raw = header[col_idx].strip() if header[col_idx] else f'col_{col_idx}'
            var_name = _sanitize_variable_name(var_name_raw)
            vals = []
            for r in data_rows:
                if col_idx >= len(r):
                    vals.append(None)
                    continue
                v = r[col_idx].strip()
                if v == '' or v == '\u00a0':
                    vals.append(None)
                    continue
                try:
                    vals.append(float(v))
                except Exception:
                    vals.append(None)
            # If hourly_like but we have far fewer rows, skip this variable
            if hourly_like and len([x for x in vals if x is not None]) < 100:
                continue
            series[var_name] = vals

        # If nothing meaningful found, return empty
        if not series:
            return {}

        # Attach metadata
        return {
            'is_hourly': hourly_like,
            'rows': len(data_rows),
            'series': series
        }
    except Exception:
        return {}


def _sanitize_variable_name(name: str) -> str:
    """Remove units in parentheses and replace spaces with underscores"""
    s = re.sub(r"\(.*?\)", '', name).strip()
    s = re.sub(r"[^0-9A-Za-z_]+", '_', s)
    s = s.strip('_')
    return s or name


def _is_number_like(s: str) -> bool:
    """Check if a string can be converted to a float"""
    try:
        float(s)
        return True
    except Exception:
        return False

In [9]:
def parse_html_with_table_lookup(html_path, log_path, file_name=None):
    """Parse EnergyPlus HTML output and log to extract structured results.
    
    Args:
        html_path: Path to the EnergyPlus HTML output file (typically output.htm or *Table.html)
        log_path: Path to the run log file (typically run_output.log) - can be None or missing
        file_name: Optional original IDF filename for reference
        
    Returns:
        Dictionary with parsed results including energy use, zones, and runtime
    """
    try:
        with open(html_path, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f, 'html.parser')
        
        # Try to read log file, but don't fail if it doesn't exist
        log_content = ""
        if log_path and Path(log_path).exists():
            with open(log_path, 'r', encoding='utf-8') as f:
                log_content = f.read()

        def get_value_from_table(table_title, row_label, col_index=1):
            """Locate a table by title and return value at row_label and column index."""
            table_header = soup.find('b', string=table_title)
            if not table_header:
                return None
            table = table_header.find_next('table')
            if not table:
                return None
            for row in table.find_all('tr'):
                cells = row.find_all('td')
                if cells and row_label in cells[0].text.strip():
                    try:
                        return float(cells[col_index].text.strip())
                    except ValueError:
                        return None
            return None

        # Extract building name
        building_name = soup.find('p', string=lambda s: s and 'Building:' in s)
        building_name = building_name.b.text if building_name and building_name.b else "Unknown Building"

        # Use provided file_name or fallback
        if not file_name:
            file_name = "unknown.idf"

        # Extract values from tables
        total_energy_use = get_value_from_table("Site and Source Energy", "Total Site Energy", 2)  # kWh/m²
        total_area = get_value_from_table("Building Area", "Total Building Area")  # m²
        
        # Extract energy values from End Uses table
        # "Heating" = space heating, "Water Systems" = domestic hot water
        heating_kwh = get_value_from_table("End Uses", "Heating", 12)  # District Heating column
        water_systems_kwh = get_value_from_table("End Uses", "Water Systems", 12)  # DHW
        cooling_kwh = get_value_from_table("End Uses", "Cooling", 12)
        lighting_kwh = get_value_from_table("End Uses", "Interior Lighting", 1)
        equipment_kwh = get_value_from_table("End Uses", "Interior Equipment", 1)

        if not total_area:
            total_area = 1.0  # fallback to avoid division by zero

        # Normalize values by area
        heating_demand = heating_kwh / total_area if heating_kwh else 0.0  # Space heating only
        dhw_demand = water_systems_kwh / total_area if water_systems_kwh else 0.0  # DHW only
        total_heating_demand = heating_demand + dhw_demand  # Combined heating
        cooling_demand = cooling_kwh / total_area if cooling_kwh else 0.0
        lighting_intensity = lighting_kwh / total_area if lighting_kwh else 0.0
        equipment_intensity = equipment_kwh / total_area if equipment_kwh else 0.0

        # Extract run time in seconds from log (if available)
        runtime_seconds = 0.0
        if log_content:
            runtime_match = re.search(r'EnergyPlus Run Time=(\d+)hr\s+(\d+)min\s+([\d\.]+)sec', log_content)
            if runtime_match:
                hr, minute, sec = map(float, runtime_match.groups())
                runtime_seconds = hr * 3600 + minute * 60 + sec

        # Extract energy use breakdown by end use
        energy_use = {}
        table_header = soup.find('b', string="End Uses")
        if table_header:
            table = table_header.find_next('table')
            if table:
                for row in table.find_all('tr'):
                    cells = row.find_all('td')
                    if len(cells) > 1:
                        end_use = cells[0].text.strip()
                        if end_use and end_use not in ["", "&nbsp;", "Total End Uses"]:
                            # Get values for electricity and district heating
                            electricity = cells[1].text.strip() if len(cells) > 1 else "0"
                            district_heating = cells[12].text.strip() if len(cells) > 12 else "0"
                            try:
                                electricity = float(electricity) if electricity else 0.0
                                district_heating = float(district_heating) if district_heating else 0.0
                                energy_use[end_use] = {
                                    "electricity": electricity,
                                    "district_heating": district_heating,
                                    "total": electricity + district_heating
                                }
                            except ValueError:
                                pass

        # Extract zone information
        # Try both "Zone Summary" (older format) and "Zone Information" (newer format)
        zones = []
        zone_table_header = soup.find('b', string="Zone Summary")
        if not zone_table_header:
            zone_table_header = soup.find('b', string="Zone Information")
        
        if zone_table_header:
            print(f"Found zone table with header: {zone_table_header.text}")
            table = zone_table_header.find_next('table')
            if table:
                rows_processed = 0
                for row in table.find_all('tr'):
                    cells = row.find_all('td')
                    # Skip header row and total rows
                    if len(cells) < 4:
                        continue
                    
                    rows_processed += 1
                    
                    # Check if this is a data row (first cell is usually a number or zone name)
                    first_cell = cells[0].text.strip()
                    if not first_cell or "Total" in first_cell:
                        continue
                    
                    try:
                        # For "Zone Summary" format: zone name in col 0, area in col 1, volume in col 4
                        # For "Zone Information" format: index in col 0, name in col 1, area in col 22, volume in col 19
                        if len(cells) > 22:  # Zone Information format
                            zone_name = cells[1].text.strip()
                            area = float(cells[22].text.strip())
                            volume = float(cells[19].text.strip())
                        else:  # Zone Summary format
                            zone_name = cells[0].text.strip()
                            area = float(cells[1].text.strip())
                            volume = float(cells[4].text.strip())
                        
                        zones.append({
                            "name": zone_name,
                            "area": area,
                            "volume": volume
                        })
                    except (ValueError, IndexError) as e:
                        # Skip rows that can't be parsed
                        print(f"Skipped row with {len(cells)} cells, first cell: {first_cell[:50] if first_cell else 'empty'}, error: {e}")
                        pass
                
                print(f"Processed {rows_processed} table rows, found {len(zones)} zones")
            else:
                print("Zone table header found but no table element after it")
        else:
            print("No zone table header found ('Zone Summary' or 'Zone Information')")

        # Combine results into a structured output
        result = {
            "building": building_name,
            "fileName": file_name,
            "totalEnergyUse": round(total_energy_use, 1) if total_energy_use else 0.0,
            "spaceHeatingDemand": round(heating_demand, 1),  # Space heating only
            "dhwDemand": round(dhw_demand, 1),  # Domestic hot water only
            "totalHeatingDemand": round(total_heating_demand, 1),  # Space + DHW
            "heatingDemand": round(heating_demand, 1),  # Keep for backwards compatibility
            "coolingDemand": round(cooling_demand, 1),
            "lightingDemand": round(lighting_intensity, 1),
            "equipmentDemand": round(equipment_intensity, 1),
            "runTime": round(runtime_seconds, 1),
            "totalArea": total_area,
            "energy_use": energy_use,
            "zones": zones,
            "status": "success"
        }

        return result

    except Exception as e:
        import traceback
        traceback.print_exc()
        
        return {
            "error": str(e),
            "fileName": file_name or "unknown.idf",
            "totalEnergyUse": 0.0,
            "heatingDemand": 0.0,
            "coolingDemand": 0.0,
            "runTime": 0.0,
            "energy_use": {},
            "zones": [],
            "status": "error"
        }

In [10]:
def process_simulation_results(output_dir, file_name=None):
    """Process EnergyPlus simulation results from a directory.
    
    Args:
        output_dir: Directory containing the simulation output files
        file_name: Optional original IDF filename for reference (e.g., "city_optimized.idf")
        
    Returns:
        Dictionary with parsed results including hourly timeseries if available
    """
    output_dir = Path(output_dir)
    
    # Try multiple naming patterns for the HTML and log files
    # Pattern 1: Standard EnergyPlus output naming
    html_path = output_dir / 'output.htm'
    log_path = output_dir / 'run_output.log'
    csv_path = output_dir / 'output.csv'
    
    # Pattern 2: If file_name is provided, derive the pattern (e.g., city_optimizedTable.html)
    if file_name and not html_path.exists():
        base_name = Path(file_name).stem  # Remove .idf extension
        html_path = output_dir / f'{base_name}Table.html'
        csv_path = output_dir / f'{base_name}.csv'
    
    # Pattern 3: Check for eplustbl.htm (original EnergyPlus name)
    if not html_path.exists():
        html_path = output_dir / 'eplustbl.htm'
    
    print(f"Looking for HTML at: {html_path}")
    print(f"Looking for log at: {log_path}")
    print(f"HTML exists: {html_path.exists()}")
    print(f"Log exists: {log_path.exists()}")
    
    if html_path.exists():
        # Extract results from HTML
        results = parse_html_with_table_lookup(html_path, log_path, file_name)

        # If a ReadVars CSV exists, attempt to parse hourly timeseries
        if csv_path.exists():
            try:
                print(f"Found CSV file at: {csv_path}")
                hourly_data = parse_readvars_csv(csv_path)
                if hourly_data:
                    results['hourly_timeseries'] = hourly_data
                    print(f"Parsed {len(hourly_data.get('series', {}))} timeseries variables")
            except Exception as e:
                print(f"Error parsing CSV: {e}")
                # Non-fatal: continue even if CSV parsing fails
                pass
        
        # Add original filename to results if provided
        if file_name:
            results['originalFileName'] = file_name
        
        # Save parsed results as a JSON file
        json_path = output_dir / 'parsed_results.json'
        with open(json_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"Saved parsed results to: {json_path}")
        
        return results
    else:
        print(f"Warning: HTML results file not found at {html_path}")
        # List available files to help debug
        try:
            available_files = [f.name for f in output_dir.iterdir() if f.is_file()]
            print(f"Available files in directory: {available_files[:10]}")  # Show first 10
        except Exception:
            pass
        
        return {
            'error': 'HTML results file not found',
            'fileName': file_name or "unknown.idf",
            'originalFileName': file_name,
            'status': 'error'
        }

## Example Usage

Now you can use these functions to parse EnergyPlus simulation results without Django dependencies.

In [11]:
# Example 1: Parse a single simulation result directory
# Using the actual output directory in notebooks
output_directory = Path(r"idf_output")

# For the optimized city building
results = process_simulation_results(output_directory, file_name="city_optimized.idf")

# View the parsed results (first few keys)
if results['status'] == 'success':
    print(f"Successfully parsed: {results['fileName']}")
    print(f"Building: {results['building']}")
    print(f"Total Energy Use: {results['totalEnergyUse']} kWh/m²")
else:
    print(f"Status: {results['status']}")
    print(f"Error: {results.get('error', 'Unknown error')}")

Looking for HTML at: idf_output\city_optimizedTable.html
Looking for log at: idf_output\run_output.log
HTML exists: True
Log exists: False
Found zone table with header: Zone Summary
Processed 72 table rows, found 67 zones
Found CSV file at: idf_output\city_optimized.csv
Parsed 255 timeseries variables
Saved parsed results to: idf_output\parsed_results.json
Successfully parsed: city_optimized.idf
Building: Unknown Building
Total Energy Use: 278.9 kWh/m²


In [12]:
# Example 2: Access specific results with separate space heating and DHW
if results['status'] == 'success':
    print(f"Building: {results['building']}")
    print(f"Total Energy Use: {results['totalEnergyUse']} kWh/m²")
    print(f"\n=== Heating Breakdown ===")
    print(f"Space Heating Demand: {results['spaceHeatingDemand']} kWh/m²")
    print(f"DHW Demand: {results['dhwDemand']} kWh/m²")
    print(f"Total Heating Demand: {results['totalHeatingDemand']} kWh/m²")
    print(f"\n=== Other ===")
    print(f"Cooling Demand: {results['coolingDemand']} kWh/m²")
    print(f"Total Area: {results['totalArea']} m²")
    print(f"Runtime: {results['runTime']} seconds")
    print(f"\nNumber of zones: {len(results['zones'])}")
    
    # Check if hourly data is available
    if 'hourly_timeseries' in results:
        hourly = results['hourly_timeseries']
        print(f"\nHourly data available: {hourly['is_hourly']}")
        print(f"Number of rows: {hourly['rows']}")
        print(f"Available variables: {list(hourly['series'].keys())[:10]}")  # First 10 variables
else:
    print(f"Error: {results.get('error', 'Unknown error')}")

Building: Unknown Building
Total Energy Use: 278.9 kWh/m²

=== Heating Breakdown ===
Space Heating Demand: 195.1 kWh/m²
DHW Demand: 38.5 kWh/m²
Total Heating Demand: 233.6 kWh/m²

=== Other ===
Cooling Demand: 0.0 kWh/m²
Total Area: 171253.45 m²
Runtime: 0.0 seconds

Number of zones: 67

Hourly data available: True
Number of rows: 8760
Available variables: ['BUILDING0_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING1_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING2_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING3_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING4_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING5_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING6_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING7_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING8_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J', 'BUILDING9_FLOOR1_ROOM1_Zone_Lights_Electricity_Energy_J']


## Building Type Analysis

Analyze results by building type (single-family vs multi-family housing, etc.)

In [15]:
def aggregate_results_by_building_type(results, geojson_path):
    """
    Aggregate energy results by building type (e.g., single-family vs multi-family).
    
    Args:
        results: Parsed simulation results dict with zones
        geojson_path: Path to enriched GeoJSON with building metadata
        
    Returns:
        Dict with aggregated results by building type
    """
    import json
    from collections import defaultdict
    
    # Load GeoJSON to get building types
    with open(geojson_path, 'r', encoding='utf-8') as f:
        geojson_data = json.load(f)
    
    # Create mapping from building ID to building type
    building_types = {}
    for feature in geojson_data.get('features', []):
        building_id = feature['properties'].get('id', '').replace('Building', '')
        andamal = feature['properties'].get('andamal1', 'Unknown')
        building_types[building_id] = andamal
    
    # Aggregate zones by building type
    type_aggregates = defaultdict(lambda: {
        'total_area': 0,
        'total_volume': 0,
        'zone_count': 0,
        'building_count': set()
    })
    
    for zone in results.get('zones', []):
        zone_name = zone['name']
        # Extract building ID from zone name (e.g., "BUILDING0_FLOOR1_ROOM1" -> "0")
        if 'BUILDING' in zone_name:
            building_id = zone_name.split('_')[0].replace('BUILDING', '')
            building_type = building_types.get(building_id, 'Unknown')
            
            type_aggregates[building_type]['total_area'] += zone['area']
            type_aggregates[building_type]['total_volume'] += zone['volume']
            type_aggregates[building_type]['zone_count'] += 1
            type_aggregates[building_type]['building_count'].add(building_id)
    
    # Convert sets to counts and calculate averages
    summary = {}
    for building_type, data in type_aggregates.items():
        summary[building_type] = {
            'building_count': len(data['building_count']),
            'zone_count': data['zone_count'],
            'total_area_m2': round(data['total_area'], 2),
            'total_volume_m3': round(data['total_volume'], 2),
            'avg_area_per_building': round(data['total_area'] / len(data['building_count']), 2) if data['building_count'] else 0
        }
    
    return summary

# Example: Aggregate by building type
geojson_path = Path(r"idf_output/city_enriched.geojson")

if geojson_path.exists():
    type_summary = aggregate_results_by_building_type(results, geojson_path)
    
    print("=== Energy Results by Building Type ===\n")
    for building_type, data in sorted(type_summary.items()):
        print(f"{building_type}:")
        print(f"  Buildings: {data['building_count']}")
        print(f"  Zones: {data['zone_count']}")
        print(f"  Total Area: {data['total_area_m2']:,.0f} m²")
        print(f"  Avg Area/Building: {data['avg_area_per_building']:,.0f} m²")
        print()
else:
    print(f"GeoJSON not found: {geojson_path}")

=== Energy Results by Building Type ===

Bostad;Flerfamiljshus:
  Buildings: 34
  Zones: 34
  Total Area: 20,642 m²
  Avg Area/Building: 607 m²

Bostad;Småhus friliggande:
  Buildings: 7
  Zones: 7
  Total Area: 969 m²
  Avg Area/Building: 138 m²

Bostad;Småhus kedjehus:
  Buildings: 13
  Zones: 13
  Total Area: 1,383 m²
  Avg Area/Building: 106 m²

Komplementbyggnad;:
  Buildings: 2
  Zones: 2
  Total Area: 374 m²
  Avg Area/Building: 187 m²

Samhällsfunktion;Ospecificerad:
  Buildings: 3
  Zones: 3
  Total Area: 2,186 m²
  Avg Area/Building: 729 m²

Samhällsfunktion;Skola:
  Buildings: 3
  Zones: 3
  Total Area: 3,073 m²
  Avg Area/Building: 1,024 m²

Samhällsfunktion;Vårdcentral:
  Buildings: 2
  Zones: 2
  Total Area: 1,363 m²
  Avg Area/Building: 681 m²

Verksamhet;:
  Buildings: 2
  Zones: 2
  Total Area: 1,894 m²
  Avg Area/Building: 947 m²

Övrig byggnad;:
  Buildings: 1
  Zones: 1
  Total Area: 261 m²
  Avg Area/Building: 261 m²



In [17]:
def categorize_building_type(andamal):
    """
    Categorize Swedish building types into broader groups.
    
    Args:
        andamal: Swedish building purpose code (e.g., "Bostad;Flerfamiljshus")
        
    Returns:
        Broad category string
    """
    if not andamal or andamal == 'Unknown':
        return 'Unknown'
    
    andamal_lower = andamal.lower()
    
    # Residential categories
    if 'småhus' in andamal_lower and 'friliggande' in andamal_lower:
        return 'Single-Family Detached'
    elif 'småhus' in andamal_lower and ('radhus' in andamal_lower or 'kedjehus' in andamal_lower):
        return 'Single-Family Attached (Row/Terraced)'
    elif 'flerfamiljshus' in andamal_lower or ('bostad' in andamal_lower and 'flera' in andamal_lower):
        return 'Multi-Family Apartment'
    elif 'bostad' in andamal_lower:
        return 'Residential (Other)'
    
    # Non-residential categories
    elif 'handel' in andamal_lower or 'butik' in andamal_lower:
        return 'Commercial/Retail'
    elif 'industri' in andamal_lower or 'lager' in andamal_lower:
        return 'Industrial/Warehouse'
    elif 'kontor' in andamal_lower:
        return 'Office'
    elif 'skola' in andamal_lower or 'förskola' in andamal_lower:
        return 'Educational'
    elif 'vård' in andamal_lower or 'sjukhus' in andamal_lower:
        return 'Healthcare'
    else:
        return 'Other'

# Test the function with a few examples
test_types = [
    'Bostad;Småhus, friliggande',
    'Bostad;Småhus, radhus',
    'Bostad;Flerfamiljshus',
    'Handel och service;Handel',
    'Industri',
    'Unknown'
]

print("Building Type Categorization Examples:")
print("-" * 60)
for test_type in test_types:
    category = categorize_building_type(test_type)
    print(f"{test_type:40s} -> {category}")

Building Type Categorization Examples:
------------------------------------------------------------
Bostad;Småhus, friliggande               -> Single-Family Detached
Bostad;Småhus, radhus                    -> Single-Family Attached (Row/Terraced)
Bostad;Flerfamiljshus                    -> Multi-Family Apartment
Handel och service;Handel                -> Commercial/Retail
Industri                                 -> Industrial/Warehouse
Unknown                                  -> Unknown


In [18]:
# Calculate heating demand for residential buildings
def calculate_residential_heating_demand(results, geojson_path):
    """
    Calculate heating demand breakdown for residential building types.
    
    Args:
        results: Parsed simulation results with heating data
        geojson_path: Path to GeoJSON with building metadata
        
    Returns:
        Dict with heating demand by residential type
    """
    import json
    from collections import defaultdict
    
    # Load building metadata
    with open(geojson_path, 'r', encoding='utf-8') as f:
        geojson_data = json.load(f)
    
    # Map building ID to type
    building_info = {}
    for feature in geojson_data['features']:
        building_id = feature['properties'].get('id', '').replace('Building', '')
        andamal = feature['properties'].get('andamal1', 'Unknown')
        building_info[building_id] = {
            'type': andamal,
            'category': categorize_building_type(andamal)
        }
    
    # Aggregate zone areas by building
    building_areas = defaultdict(float)
    for zone in results.get('zones', []):
        zone_name = zone['name']
        if 'BUILDING' in zone_name:
            building_id = zone_name.split('_')[0].replace('BUILDING', '')
            building_areas[building_id] += zone['area']
    
    # Calculate heating demand per building (assuming uniform distribution)
    # Total heating from results (kWh/m²)
    space_heating_intensity = results.get('spaceHeatingDemand', 0)  # kWh/m²
    dhw_intensity = results.get('dhwDemand', 0)  # kWh/m²
    total_heating_intensity = results.get('totalHeatingDemand', 0)  # kWh/m²
    
    # Aggregate by residential category
    residential_summary = defaultdict(lambda: {
        'building_count': 0,
        'total_area_m2': 0,
        'space_heating_kwh': 0,
        'dhw_kwh': 0,
        'total_heating_kwh': 0
    })
    
    for building_id, area in building_areas.items():
        if building_id in building_info:
            category = building_info[building_id]['category']
            building_type = building_info[building_id]['type']
            
            # Filter for residential only
            if 'Bostad' in building_type or 'Family' in category:
                # Calculate heating for this building
                space_heating = space_heating_intensity * area
                dhw = dhw_intensity * area
                total_heating = total_heating_intensity * area
                
                residential_summary[category]['building_count'] += 1
                residential_summary[category]['total_area_m2'] += area
                residential_summary[category]['space_heating_kwh'] += space_heating
                residential_summary[category]['dhw_kwh'] += dhw
                residential_summary[category]['total_heating_kwh'] += total_heating
    
    # Calculate intensities (kWh/m²)
    for category, data in residential_summary.items():
        if data['total_area_m2'] > 0:
            data['space_heating_kwh_m2'] = round(data['space_heating_kwh'] / data['total_area_m2'], 1)
            data['dhw_kwh_m2'] = round(data['dhw_kwh'] / data['total_area_m2'], 1)
            data['total_heating_kwh_m2'] = round(data['total_heating_kwh'] / data['total_area_m2'], 1)
            data['dhw_percentage'] = round((data['dhw_kwh'] / data['total_heating_kwh'] * 100), 1) if data['total_heating_kwh'] > 0 else 0
    
    return dict(residential_summary)

# Run the analysis
if geojson_path.exists() and results.get('status') == 'success':
    residential_heating = calculate_residential_heating_demand(results, geojson_path)
    
    print("=" * 80)
    print("HEATING DEMAND ANALYSIS FOR RESIDENTIAL BUILDINGS")
    print("=" * 80)
    print(f"\nSimulation Area-Averaged Values:")
    print(f"  Space Heating: {results.get('spaceHeatingDemand', 0)} kWh/m²")
    print(f"  DHW: {results.get('dhwDemand', 0)} kWh/m²")
    print(f"  Total Heating: {results.get('totalHeatingDemand', 0)} kWh/m²")
    print("\n" + "=" * 80)
    
    if residential_heating:
        # Calculate totals
        total_buildings = sum(d['building_count'] for d in residential_heating.values())
        total_area = sum(d['total_area_m2'] for d in residential_heating.values())
        total_heating_energy = sum(d['total_heating_kwh'] for d in residential_heating.values())
        
        for category in sorted(residential_heating.keys()):
            data = residential_heating[category]
            print(f"\n{category.upper()}")
            print("-" * 80)
            print(f"  Number of Buildings:  {data['building_count']}")
            print(f"  Total Floor Area:     {data['total_area_m2']:,.0f} m² ({data['total_area_m2']/total_area*100:.1f}% of residential)")
            print(f"\n  Energy Intensities:")
            print(f"    Space Heating:      {data['space_heating_kwh_m2']} kWh/m²·year")
            print(f"    DHW:                {data['dhw_kwh_m2']} kWh/m²·year ({data['dhw_percentage']}% of total)")
            print(f"    Total Heating:      {data['total_heating_kwh_m2']} kWh/m²·year")
            print(f"\n  Total Annual Energy:")
            print(f"    Space Heating:      {data['space_heating_kwh']:,.0f} kWh/year")
            print(f"    DHW:                {data['dhw_kwh']:,.0f} kWh/year")
            print(f"    Total Heating:      {data['total_heating_kwh']:,.0f} kWh/year ({data['total_heating_kwh']/total_heating_energy*100:.1f}% of residential)")
        
        print("\n" + "=" * 80)
        print(f"TOTAL RESIDENTIAL STOCK:")
        print(f"  Buildings: {total_buildings}")
        print(f"  Floor Area: {total_area:,.0f} m²")
        print(f"  Annual Heating Energy: {total_heating_energy:,.0f} kWh/year")
        print("=" * 80)
    else:
        print("\nNo residential buildings found in the simulation.")
else:
    print("Results or GeoJSON not available. Run the simulation parsing first.")

HEATING DEMAND ANALYSIS FOR RESIDENTIAL BUILDINGS

Simulation Area-Averaged Values:
  Space Heating: 195.1 kWh/m²
  DHW: 38.5 kWh/m²
  Total Heating: 233.6 kWh/m²


MULTI-FAMILY APARTMENT
--------------------------------------------------------------------------------
  Number of Buildings:  34
  Total Floor Area:     20,642 m² (89.8% of residential)

  Energy Intensities:
    Space Heating:      195.1 kWh/m²·year
    DHW:                38.5 kWh/m²·year (16.5% of total)
    Total Heating:      233.6 kWh/m²·year

  Total Annual Energy:
    Space Heating:      4,027,352 kWh/year
    DHW:                794,736 kWh/year
    Total Heating:      4,822,088 kWh/year (89.8% of residential)

SINGLE-FAMILY ATTACHED (ROW/TERRACED)
--------------------------------------------------------------------------------
  Number of Buildings:  13
  Total Floor Area:     1,383 m² (6.0% of residential)

  Energy Intensities:
    Space Heating:      195.1 kWh/m²·year
    DHW:                38.5 kWh/m²·year 

## Zone-Level Heating Analysis

The uniform heating demand across building types suggests that all buildings were assigned the same program type in the IDF. Let's investigate by:
1. Checking what building programs are actually used in the IDF
2. Parsing zone-level heating data from the CSV to see if individual zones have different demands

In [23]:
def check_idf_building_programs(idf_path):
    """
    Check what building program types are actually used in the IDF file.
    
    Args:
        idf_path: Path to IDF file
        
    Returns:
        Dict with program type counts
    """
    import re
    from collections import Counter
    
    try:
        with open(idf_path, 'r', encoding='utf-8') as f:
            idf_content = f.read()
        
        # Find all program type assignments in the IDF
        # Pattern: 2019::BUILDINGTYPE::SPACETYPE
        program_pattern = r'2019::(\w+)::(\w+)'
        matches = re.findall(program_pattern, idf_content, re.IGNORECASE)
        
        if matches:
            # Count unique program types
            program_counts = Counter([f"{building_type}::{space_type}" for building_type, space_type in matches])
            
            print("=" * 80)
            print("BUILDING PROGRAMS FOUND IN IDF")
            print("=" * 80)
            for program, count in program_counts.most_common():
                print(f"  {program}: {count} occurrences")
            print("=" * 80)
            
            return dict(program_counts)
        else:
            print("No building program types found in IDF")
            return {}
            
    except Exception as e:
        print(f"Error reading IDF: {e}")
        return {}

# Check the IDF file
idf_path = output_directory / "city_optimized.idf"
if idf_path.exists():
    program_types = check_idf_building_programs(idf_path)
    
    if len(program_types) == 1:
        print("\n⚠️  WARNING: All buildings use the SAME program type!")
        print("This explains why heating demand is identical across building types.")
        print("\nTo get different heating demands, you need to:")
        print("  1. Check geojson_to_idf.py assigns different programs based on andamal1 field")
        print("  2. Verify BUILDING_PROGRAM_DICT mapping is being applied")
        print("  3. Re-run IDF generation if needed")
else:
    print(f"IDF file not found: {idf_path}")

BUILDING PROGRAMS FOUND IN IDF
  HighriseApartment::Apartment_SHW: 536 occurrences
  HighriseApartment::Apartment_Setpoint: 134 occurrences
  HighriseApartment::Apartment_Ventilation: 134 occurrences
  HighriseApartment::Apartment_People: 67 occurrences
  HighriseApartment::Apartment_Lighting: 67 occurrences
  HighriseApartment::Apartment_Electric: 67 occurrences
  HighriseApartment::Apartment_Infiltration: 67 occurrences


In [24]:
def parse_zone_level_heating(csv_path, geojson_path):
    """
    Parse zone-level heating demand from CSV timeseries to see actual variation.
    
    Args:
        csv_path: Path to EnergyPlus output CSV
        geojson_path: Path to GeoJSON with building metadata
        
    Returns:
        Dict with heating demand by building ID
    """
    import csv
    import json
    import re
    from collections import defaultdict
    
    try:
        print("Parsing zone-level heating data from CSV...")
        
        # Load building metadata
        with open(geojson_path, 'r', encoding='utf-8') as f:
            geojson_data = json.load(f)
        
        building_types = {}
        for feature in geojson_data['features']:
            building_id = feature['properties'].get('id', '').replace('Building', '')
            andamal = feature['properties'].get('andamal1', 'Unknown')
            building_types[building_id] = {
                'type': andamal,
                'category': categorize_building_type(andamal)
            }
        
        # Parse CSV header to find heating columns
        with open(csv_path, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            header = next(reader)
            
            # Find columns for zone heating and DHW
            heating_cols = {}
            dhw_cols = {}
            
            for idx, col_name in enumerate(header):
                if 'Zone Ideal Loads Supply Air Total Heating Energy' in col_name:
                    # Extract building and zone info
                    match = re.search(r'BUILDING(\d+)_', col_name)
                    if match:
                        building_id = match.group(1)
                        heating_cols[building_id] = idx
                
                elif 'Water Use Equipment Heating Energy' in col_name:
                    # DHW heating
                    match = re.search(r'BUILDING(\d+)_', col_name)
                    if match:
                        building_id = match.group(1)
                        dhw_cols[building_id] = idx
            
            print(f"Found {len(heating_cols)} buildings with zone heating data")
            print(f"Found {len(dhw_cols)} buildings with DHW data")
            
            # Sum annual energy by building
            building_heating = defaultdict(lambda: {'space_heating_j': 0, 'dhw_j': 0})
            
            row_count = 0
            for row in reader:
                row_count += 1
                if row_count > 8760:  # One year of hourly data
                    break
                    
                for building_id, col_idx in heating_cols.items():
                    try:
                        value = float(row[col_idx])
                        building_heating[building_id]['space_heating_j'] += value
                    except (ValueError, IndexError):
                        pass
                
                for building_id, col_idx in dhw_cols.items():
                    try:
                        value = float(row[col_idx])
                        building_heating[building_id]['dhw_j'] += value
                    except (ValueError, IndexError):
                        pass
        
        print(f"Processed {row_count} hourly timesteps")
        
        # Convert to kWh and link to building types
        building_results = {}
        for building_id, energy in building_heating.items():
            # J to kWh: divide by 3.6e6
            space_heating_kwh = energy['space_heating_j'] / 3.6e6
            dhw_kwh = energy['dhw_j'] / 3.6e6
            
            # Get building area from results dict
            building_area = 0
            for zone in results.get('zones', []):
                if f'BUILDING{building_id}_' in zone['name']:
                    building_area += zone['area']
            
            if building_area > 0 and building_id in building_types:
                building_results[building_id] = {
                    'building_type': building_types[building_id]['type'],
                    'category': building_types[building_id]['category'],
                    'area_m2': building_area,
                    'space_heating_kwh': space_heating_kwh,
                    'dhw_kwh': dhw_kwh,
                    'total_heating_kwh': space_heating_kwh + dhw_kwh,
                    'space_heating_kwh_m2': round(space_heating_kwh / building_area, 1),
                    'dhw_kwh_m2': round(dhw_kwh / building_area, 1),
                    'total_heating_kwh_m2': round((space_heating_kwh + dhw_kwh) / building_area, 1)
                }
        
        print(f"\nSuccessfully parsed {len(building_results)} buildings")
        return building_results
        
    except Exception as e:
        print(f"Error parsing zone-level heating: {e}")
        import traceback
        traceback.print_exc()
        return {}

# Parse zone-level data
csv_path = output_directory / "city_optimized.csv"
if csv_path.exists() and geojson_path.exists():
    zone_heating_data = parse_zone_level_heating(csv_path, geojson_path)
    
    if zone_heating_data:
        # Show first few buildings
        print("\n" + "=" * 80)
        print("SAMPLE ZONE-LEVEL HEATING DATA (First 10 Buildings)")
        print("=" * 80)
        for i, (building_id, data) in enumerate(list(zone_heating_data.items())[:10]):
            print(f"\nBuilding {building_id} - {data['category']}")
            print(f"  Type: {data['building_type']}")
            print(f"  Area: {data['area_m2']:.0f} m²")
            print(f"  Space Heating: {data['space_heating_kwh_m2']} kWh/m²")
            print(f"  DHW: {data['dhw_kwh_m2']} kWh/m²")
            print(f"  Total: {data['total_heating_kwh_m2']} kWh/m²")
else:
    print("CSV or GeoJSON not found")

Parsing zone-level heating data from CSV...
Found 27 buildings with zone heating data
Found 67 buildings with DHW data
Processed 8760 hourly timesteps

Successfully parsed 67 buildings

SAMPLE ZONE-LEVEL HEATING DATA (First 10 Buildings)

Building 0 - Educational
  Type: Samhällsfunktion;Skola
  Area: 835 m²
  Space Heating: 751.6 kWh/m²
  DHW: 154.2 kWh/m²
  Total: 905.8 kWh/m²

Building 1 - Multi-Family Apartment
  Type: Bostad;Flerfamiljshus
  Area: 408 m²
  Space Heating: 860.9 kWh/m²
  DHW: 154.2 kWh/m²
  Total: 1015.0 kWh/m²

Building 2 - Single-Family Detached
  Type: Bostad;Småhus friliggande
  Area: 176 m²
  Space Heating: 380.3 kWh/m²
  DHW: 77.1 kWh/m²
  Total: 457.4 kWh/m²

Building 3 - Single-Family Detached
  Type: Bostad;Småhus friliggande
  Area: 161 m²
  Space Heating: 388.7 kWh/m²
  DHW: 77.1 kWh/m²
  Total: 465.8 kWh/m²

Building 4 - Single-Family Detached
  Type: Bostad;Småhus friliggande
  Area: 161 m²
  Space Heating: 371.3 kWh/m²
  DHW: 77.1 kWh/m²
  Total: 448.4

In [25]:
# Aggregate zone-level heating by residential building type
def aggregate_zone_heating_by_type(zone_heating_data):
    """Aggregate zone-level heating data by residential building category."""
    from collections import defaultdict
    
    type_aggregates = defaultdict(lambda: {
        'building_count': 0,
        'total_area_m2': 0,
        'space_heating_kwh': 0,
        'dhw_kwh': 0,
        'total_heating_kwh': 0
    })
    
    for building_id, data in zone_heating_data.items():
        category = data['category']
        building_type = data['building_type']
        
        # Filter for residential only
        if 'Bostad' in building_type or 'Family' in category:
            type_aggregates[category]['building_count'] += 1
            type_aggregates[category]['total_area_m2'] += data['area_m2']
            type_aggregates[category]['space_heating_kwh'] += data['space_heating_kwh']
            type_aggregates[category]['dhw_kwh'] += data['dhw_kwh']
            type_aggregates[category]['total_heating_kwh'] += data['total_heating_kwh']
    
    # Calculate intensities
    summary = {}
    for category, data in type_aggregates.items():
        if data['total_area_m2'] > 0:
            summary[category] = {
                'building_count': data['building_count'],
                'total_area_m2': round(data['total_area_m2'], 0),
                'space_heating_kwh_m2': round(data['space_heating_kwh'] / data['total_area_m2'], 1),
                'dhw_kwh_m2': round(data['dhw_kwh'] / data['total_area_m2'], 1),
                'total_heating_kwh_m2': round(data['total_heating_kwh'] / data['total_area_m2'], 1),
                'space_heating_kwh': round(data['space_heating_kwh'], 0),
                'dhw_kwh': round(data['dhw_kwh'], 0),
                'total_heating_kwh': round(data['total_heating_kwh'], 0),
                'dhw_percentage': round((data['dhw_kwh'] / data['total_heating_kwh'] * 100), 1) if data['total_heating_kwh'] > 0 else 0
            }
    
    return summary

if zone_heating_data:
    residential_zone_heating = aggregate_zone_heating_by_type(zone_heating_data)
    
    print("\n" + "=" * 80)
    print("ZONE-LEVEL RESIDENTIAL HEATING ANALYSIS")
    print("=" * 80)
    
    if residential_zone_heating:
        total_buildings = sum(d['building_count'] for d in residential_zone_heating.values())
        total_area = sum(d['total_area_m2'] for d in residential_zone_heating.values())
        total_heating = sum(d['total_heating_kwh'] for d in residential_zone_heating.values())
        
        for category in sorted(residential_zone_heating.keys()):
            data = residential_zone_heating[category]
            print(f"\n{category.upper()}")
            print("-" * 80)
            print(f"  Number of Buildings:  {data['building_count']}")
            print(f"  Total Floor Area:     {data['total_area_m2']:,.0f} m² ({data['total_area_m2']/total_area*100:.1f}% of residential)")
            print(f"\n  Energy Intensities:")
            print(f"    Space Heating:      {data['space_heating_kwh_m2']} kWh/m²·year")
            print(f"    DHW:                {data['dhw_kwh_m2']} kWh/m²·year ({data['dhw_percentage']}% of total)")
            print(f"    Total Heating:      {data['total_heating_kwh_m2']} kWh/m²·year")
            print(f"\n  Total Annual Energy:")
            print(f"    Space Heating:      {data['space_heating_kwh']:,.0f} kWh/year")
            print(f"    DHW:                {data['dhw_kwh']:,.0f} kWh/year")
            print(f"    Total Heating:      {data['total_heating_kwh']:,.0f} kWh/year ({data['total_heating_kwh']/total_heating*100:.1f}% of residential)")
        
        print("\n" + "=" * 80)
        print(f"TOTAL RESIDENTIAL STOCK:")
        print(f"  Buildings: {total_buildings}")
        print(f"  Floor Area: {total_area:,.0f} m²")
        print(f"  Annual Heating Energy: {total_heating:,.0f} kWh/year")
        print("=" * 80)
        
        # Compare to original analysis
        print("\n" + "=" * 80)
        print("COMPARISON: Zone-Level vs. Area-Averaged Analysis")
        print("=" * 80)
        print("\nThis shows whether heating demands actually vary by building type,")
        print("or if the uniform values are due to identical program assignments.")
        print("=" * 80)
    else:
        print("\nNo residential buildings found")
else:
    print("Run zone-level parsing first")


ZONE-LEVEL RESIDENTIAL HEATING ANALYSIS

MULTI-FAMILY APARTMENT
--------------------------------------------------------------------------------
  Number of Buildings:  34
  Total Floor Area:     20,642 m² (89.8% of residential)

  Energy Intensities:
    Space Heating:      267.1 kWh/m²·year
    DHW:                250.0 kWh/m²·year (48.3% of total)
    Total Heating:      517.1 kWh/m²·year

  Total Annual Energy:
    Space Heating:      5,513,559 kWh/year
    DHW:                5,160,428 kWh/year
    Total Heating:      10,673,987 kWh/year (90.2% of residential)

SINGLE-FAMILY ATTACHED (ROW/TERRACED)
--------------------------------------------------------------------------------
  Number of Buildings:  13
  Total Floor Area:     1,383 m² (6.0% of residential)

  Energy Intensities:
    Space Heating:      412.5 kWh/m²·year
    DHW:                98.4 kWh/m²·year (19.3% of total)
    Total Heating:      510.9 kWh/m²·year

  Total Annual Energy:
    Space Heating:      570,256 kWh/